In [0]:
%sql
SELECT current_catalog(), current_schema();

current_catalog(),current_schema()
workspace,default


In [0]:
import urllib.request
url = "https://github.com/StackExchange/Survey/raw/refs/heads/main/packages/archive/2025/schema.csv"
try:
    data = urllib.request.urlopen(url, timeout=30).read()
    print("OK, octets reçus :", len(data))
except Exception as e:
    print("Échec :", repr(e))

OK, octets reçus : 30403


In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.ai_reality_check;


In [0]:
%sql
CREATE VOLUME IF NOT EXISTS workspace.ai_reality_check.raw;

In [0]:
import urllib.request, hashlib, os

BASE = "https://github.com/StackExchange/Survey/raw/refs/heads/main/packages/archive/2025"
DEST = "/Volumes/workspace/ai_reality_check/raw/survey_2025"
os.makedirs(DEST, exist_ok=True)

EXPECTED = {
    "schema.csv": (30403, "1d24951e04eab46c6f9fecef6ce8e6b32a0a14a3f0eecdcf62f70833a74b3ff8"),
    "results.csv": (140893245, "2d1f65308877282edfb4470520eabbc08cb499118432a3dcec6a66c086aa2baa"),
}

def download(name, expected_size, expected_sha):
    h, size = hashlib.sha256(), 0
    with urllib.request.urlopen(f"{BASE}/{name}", timeout=120) as r, open(f"{DEST}/{name}", "wb") as f:
        while chunk := r.read(1024 * 1024):
            f.write(chunk)
            h.update(chunk)
            size += len(chunk)
    ok = size == expected_size and h.hexdigest() == expected_sha
    print(f"{name}: {size:,} octets | SHA-256 {'identique' if ok else 'DIFFÉRENT'}")
    return ok

results = [download(n, s, sha) for n, (s, sha) in EXPECTED.items()]
print("Tous les fichiers sont valides :", all(results))

schema.csv: 30,403 octets | SHA-256 identique
results.csv: 140,893,245 octets | SHA-256 identique
Tous les fichiers sont valides : True


In [0]:
display(dbutils.fs.ls("/Volumes/workspace/ai_reality_check/raw/survey_2025"))

path,name,size,modificationTime
dbfs:/Volumes/workspace/ai_reality_check/raw/survey_2025/results.csv,results.csv,140893245,1792935924000
dbfs:/Volumes/workspace/ai_reality_check/raw/survey_2025/schema.csv,schema.csv,30403,1792935923000
